# Sensitivity of the 0.92 cutoff, and what the free signals actually cost

Reviewer 2 asks for two numbers the paper argues for but never measures: how the
duplicate test behaves when its cosine cutoff moves, and what the signals cost to run.

Both are computed here from the released corpus at the submission tag, with the signal
code copied verbatim from `code/signals.py` so that nothing is reimplemented by hand.
The run is CPU only and needs no annotation.

The first thing the notebook does is reproduce the paper's figures at 0.92. If that
check fails, every number below it is worthless, so it asserts rather than prints.

In [ ]:
import ast, json, re, time, urllib.request
import numpy as np

BASE = ("https://raw.githubusercontent.com/achenna-source/"
        "icv-agent-observability/neucom-d-26-17269-r1/")

def fetch(path):
    return urllib.request.urlopen(BASE + path, timeout=90).read().decode("utf-8")

rows = [json.loads(l) for l in fetch("data/trajectories_200.jsonl").splitlines() if l.strip()]
print("trajectories: %d" % len(rows))

def steps(r):
    t = r["trajectory"]
    return ast.literal_eval(t) if isinstance(t, str) else t

TRAJ = [steps(r) for r in rows]
HUMAN = [r["label_human"] for r in rows]
FAIL = [h != "S" for h in HUMAN]
NF, NS = sum(FAIL), len(FAIL) - sum(FAIL)
print("failures %d, successes %d" % (NF, NS))

In [ ]:
# the released signal code, fetched rather than retyped
src = fetch("code/signals.py")
ns = {}
exec(compile(src, "signals.py", "exec"), ns)
tier1, tier1_exempt, tier2 = ns["tier1"], ns["tier1_exempt"], ns["tier2"]
print("loaded %d bytes of signals.py" % len(src))

from sentence_transformers import SentenceTransformer
t0 = time.perf_counter()
enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
load_s = time.perf_counter() - t0
print("encoder loaded in %.1f s (one-off)" % load_s)

In [ ]:
# embed every step once; the sweep then only recomputes cosines
def texts_of(tr):
    return [f"{s.get('thought','')} {s.get('action','')} {s.get('input','')}".strip()
            for s in tr]

t0 = time.perf_counter()
EMB = [enc.encode(texts_of(tr), normalize_embeddings=True) if len(tr) >= 2 else None
       for tr in TRAJ]
embed_s = time.perf_counter() - t0
nsteps = sum(len(tr) for tr in TRAJ)
print("embedded %d steps over %d trajectories in %.2f s" % (nsteps, len(TRAJ), embed_s))

MAXSIM = []
for e in EMB:
    if e is None or len(e) < 2:
        MAXSIM.append(0.0)
        continue
    g = e @ e.T
    np.fill_diagonal(g, -1.0)
    MAXSIM.append(float(g.max()))
MAXSIM = np.array(MAXSIM)
print("max pairwise cosine: min %.3f, median %.3f, max %.3f"
      % (MAXSIM.min(), np.median(MAXSIM), MAXSIM.max()))

In [ ]:
# contradiction test and schema validation do not depend on the cutoff
CONTRA = [tier2(tr, encoder=None)[2] for tr in TRAJ]
B = [tier1(tr)[0] for tr in TRAJ]
B_EX = [tier1_exempt(tr)[0] for tr in TRAJ]

def score(flags):
    tp = sum(1 for f, k in zip(flags, FAIL) if f and k)
    fp = sum(1 for f, k in zip(flags, FAIL) if f and not k)
    prec = tp / (tp + fp) if tp + fp else float("nan")
    return tp, fp, prec, tp / NF

# the paper's configuration, at the published cutoff
dup92 = MAXSIM > 0.92
C92 = [d or c for d, c in zip(dup92, CONTRA)]
U92 = [b or c for b, c in zip(B, C92)]
for name, flags in (("Signal B", B), ("Signal C", C92), ("B or C", U92)):
    tp, fp, prec, cov = score(flags)
    print("%-9s tp %3d  fp %3d  precision %.3f  coverage %.3f" % (name, tp, fp, prec, cov))

cov_u = score(U92)[3]
assert abs(round(cov_u, 3) - 0.531) <= 0.001, "union coverage is %.3f, paper says 0.531" % cov_u
print("\nCHECK: union coverage reproduces the paper's 0.531")

In [ ]:
# the sweep the reviewer asks for
print("cutoff   dup   C:tp  fp  prec   cov    B or C:tp  fp  prec   cov")
SWEEP = []
for th in [0.80, 0.85, 0.88, 0.90, 0.92, 0.95, 0.98, 0.99]:
    dup = MAXSIM > th
    c = [d or k for d, k in zip(dup, CONTRA)]
    u = [b or k for b, k in zip(B, c)]
    ctp, cfp, cpr, ccov = score(c)
    utp, ufp, upr, ucov = score(u)
    SWEEP.append((th, int(dup.sum()), ctp, cfp, cpr, ccov, utp, ufp, upr, ucov))
    print("%.2f    %3d    %3d  %3d  %.3f  %.3f      %3d  %3d  %.3f  %.3f"
          % (th, dup.sum(), ctp, cfp, cpr, ccov, utp, ufp, upr, ucov))

flat = [s for s in SWEEP if s[1] == SWEEP[0][1]]
print("\ntrajectories the duplicate test flags at 0.80 and at 0.99: %d and %d"
      % (SWEEP[0][1], SWEEP[-1][1]))
print("range of union coverage across the whole sweep: %.3f to %.3f"
      % (min(s[9] for s in SWEEP), max(s[9] for s in SWEEP)))

In [ ]:
# where the cutoff would have to sit to change anything
order = np.sort(MAXSIM)[::-1]
print("ten highest max-pairwise similarities in the corpus:")
print("   " + "  ".join("%.3f" % v for v in order[:10]))
print("\nnumber of trajectories above each cutoff:")
for th in [0.80, 0.85, 0.90, 0.92, 0.95]:
    print("   > %.2f : %d" % (th, int((MAXSIM > th).sum())))

In [ ]:
# what the signals cost per trajectory, on CPU
REP = 5
t0 = time.perf_counter()
for _ in range(REP):
    for tr in TRAJ:
        tier1(tr)
b_ms = (time.perf_counter() - t0) / REP / len(TRAJ) * 1000

t0 = time.perf_counter()
for _ in range(REP):
    for tr in TRAJ:
        tier2(tr, encoder=None)
contra_ms = (time.perf_counter() - t0) / REP / len(TRAJ) * 1000

t0 = time.perf_counter()
for tr in TRAJ:
    if len(tr) >= 2:
        e = enc.encode(texts_of(tr), normalize_embeddings=True)
        g = e @ e.T
        np.fill_diagonal(g, -1.0)
        float(g.max())
dup_ms = (time.perf_counter() - t0) / len(TRAJ) * 1000

print("per trajectory, averaged over %d trajectories of %.1f steps on CPU"
      % (len(TRAJ), nsteps / len(TRAJ)))
print("   schema validation (Signal B)      %7.3f ms" % b_ms)
print("   contradiction test (Signal C)     %7.3f ms" % contra_ms)
print("   duplicate test, encode + cosines  %7.3f ms" % dup_ms)
print("   total                             %7.3f ms" % (b_ms + contra_ms + dup_ms))
print("\nencoder load, paid once per process: %.1f s" % load_s)
print("model parameters: 22.7M (all-MiniLM-L6-v2)")
import platform, os
print("\nhost: %s, %d logical CPUs" % (platform.processor() or platform.machine(),
                                      os.cpu_count()))

In [ ]:
print("=== what goes into the manuscript ===\n")
print("sweep, union of B and C:")
for th, nd, ctp, cfp, cpr, ccov, utp, ufp, upr, ucov in SWEEP:
    print("   cutoff %.2f -> duplicate flags %d, union coverage %.3f, union precision %.3f"
          % (th, nd, ucov, upr))
print()
print("cost per trajectory: B %.3f ms, contradiction %.3f ms, duplicate %.3f ms, "
      "total %.3f ms" % (b_ms, contra_ms, dup_ms, b_ms + contra_ms + dup_ms))
print("encoder load %.1f s once per process, 22.7M parameters, CPU only" % load_s)